In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from the_well.data import WellDataset
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

from modules import *
from datasets import HelmholtzDataset

In [ ]:
SEED       = 42
EPOCHS     = 100
BATCH_SIZE = 8

MODES1     = 16   # Modos de Fourier na primeira dimensão espacial (x)
MODES2     = 16   # Modos de Fourier na segunda dimensão espacial (y)
WIDTH      = 32   # Número de canais internos (largura do modelo)
DEPTH      = 4    # Quantidade de camadas de Fourier
PROJ_DIM   = 128  # Dimensão da MLP de projeção para a saída

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
print(f"Torch: {torch.__version__}  |  Torchvision: {__import__('torchvision').__version__}")

In [ ]:
train_dataset = WellDataset(
    path="/mnt/storage_C1/BILL_pino/the_well/datasets/helmholtz_staircase",
    well_split_name="train"
)

validation_dataset = WellDataset(
    path="/mnt/storage_C1/BILL_pino/the_well/datasets/helmholtz_staircase",
    well_split_name="valid"
)

train_ds = HelmholtzDataset(train_dataset)
val_ds = HelmholtzDataset(validation_dataset)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=16,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=16,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

In [ ]:
x, y = train_ds[0]
in_dim, out_dim = x.shape[-1], y.shape[-1] 

model = FNO2d(
    modes1=MODES1,
    modes2=MODES2,
    width=WIDTH,
    in_dim=in_dim,
    out_dim=out_dim,
    depth=DEPTH,
    proj_dim=PROJ_DIM
).cuda()
model = torch.compile(model)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

criterion = relative_l2_loss

In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=EPOCHS
)